# Training Neural Networks & Exporting Weights/Biases

This notebook is used on a personal computer to train the NN and export weights/biases to transfer to AMD AUP-ZU3 board.

## Import Required Libraries

Will make a venv to activate for installing proper dependencies.

In [20]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')

## Load and Preprocess MNIST Dataset

In [22]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to range [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten 28x28 images to 784-dimensional vectors
x_train_flat = x_train.reshape(-1, 28*28)
x_test_flat = x_test.reshape(-1, 28*28)

# Convert labels to one-hot encoding
y_train_encoded = to_categorical(y_train, 10)
y_test_encoded = to_categorical(y_test, 10)

print(f"Training set shape: {x_train_flat.shape}")
print(f"Test set shape: {x_test_flat.shape}")
print(f"Training labels shape: {y_train_encoded.shape}")

Training set shape: (60000, 784)
Test set shape: (10000, 784)
Training labels shape: (60000, 10)


## Build Neural Network Model

Create a fully connected neural network with:
- Input layer: 784 neurons (28×28 pixels)
- Hidden layer: 128 neurons with ReLU activation
- Output layer: 10 neurons (one for each digit) with softmax activation

In [23]:
# Build the neural network model
# Architecture: 784 -> 128 -> 10 (matches HLS hardware design)
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

## Train the Model

In [24]:
# Train the model
history = model.fit(
    x_train_flat, y_train_encoded,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print("\nTraining complete!")

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8915 - loss: 0.3910 - val_accuracy: 0.9513 - val_loss: 0.1776
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9489 - loss: 0.1789 - val_accuracy: 0.9653 - val_loss: 0.1307
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9623 - loss: 0.1298 - val_accuracy: 0.9712 - val_loss: 0.1031
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9708 - loss: 0.0995 - val_accuracy: 0.9740 - val_loss: 0.0941
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9768 - loss: 0.0797 - val_accuracy: 0.9765 - val_loss: 0.0839
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9811 - loss: 0.0655 - val_accuracy: 0.9770 - val_loss: 0.0782
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9848 - loss: 0.0547 - val_accuracy: 0.9772 - val_loss: 0.0765
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9869 - loss: 0.0461 - val_accuracy: 0.

## Evaluate Model Performance

In [16]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(x_test_flat, y_test_encoded, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

Test Loss: 0.0752
Test Accuracy: 0.9763 (97.63%)


## Save the Trained Model

Save the trained model for future use without needing to retrain.

In [19]:
# Extract model weights and biases
weights = []
biases = []

for layer in model.layers:
    if hasattr(layer, 'kernel'):
        w, b = layer.get_weights()
        weights.append(w)
        biases.append(b)

print(f"Extracted {len(weights)} layers of weights")
for i, w in enumerate(weights):
    print(f"  Layer {i}: weights shape {w.shape}, biases shape {biases[i].shape}")

# Save the Keras model
model.save('mnist_model.keras')
print("Model saved as 'mnist_model.keras'")

# Export weights as .npy files for numpy inference on the PYNQ board
np.save('w1.npy', weights[0])  # (784, 128)
np.save('b1.npy', biases[0])   # (128,)
np.save('w2.npy', weights[1])  # (128, 10)
np.save('b2.npy', biases[1])   # (10,)
print("Weights exported: w1.npy, b1.npy, w2.npy, b2.npy")

Extracted 2 layers of weights
  Layer 0: weights shape (784, 128), biases shape (128,)
  Layer 1: weights shape (128, 10), biases shape (10,)
Model saved as 'mnist_model.keras'
Weights exported: w1.npy, b1.npy, w2.npy, b2.npy
